In [1]:
!pip install --quiet --upgrade numpy scipy scikit-learn mlflow


In [2]:
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("bonsai_diagnosis")

df = pd.read_csv("data/bonsai_symptoms.csv")
df.head()

,leaf_color,soil_moisture_pct,light_hours_per_day,days_since_watered,pest_sighted,likely_cause
0,green,89.7,9.0,2,0,overwatering
1,green,39.4,5.9,3,0,healthy
2,green,36.3,1.4,5,0,low_light
3,brown,15.3,5.1,7,0,underwatering
4,pale_green,51.3,3.0,1,0,low_light


In [3]:
categorical_features = ["leaf_color"]
numeric_features = ["soil_moisture_pct", "light_hours_per_day", "days_since_watered", "pest_sighted"]

X = df[categorical_features + numeric_features]
y = df["likely_cause"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
], remainder="passthrough")

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=5, random_state=42))
])

with mlflow.start_run(run_name="decision_tree_v2"):
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    acc = accuracy_score(y_test, preds)

    mlflow.log_param("max_depth", 5)
    mlflow.log_param("model_type", "DecisionTreeClassifier")
    mlflow.log_metric("accuracy", acc)
    mlflow.sklearn.log_model(pipeline, "model", serialization_format="pickle")

    print("Accuracy:", acc)
    print(classification_report(y_test, preds))

2026/09/30 02:12:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/30 02:12:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Accuracy: 0.85
               precision    recall  f1-score   support

      healthy       0.63      0.86      0.73        14
    low_light       0.86      1.00      0.92        12
 overwatering       1.00      0.56      0.71         9
         pest       1.00      0.70      0.82        10
underwatering       1.00      1.00      1.00        15

     accuracy                           0.85        60
    macro avg       0.90      0.82      0.84        60
 weighted avg       0.89      0.85      0.85        60

🏃 View run decision_tree_v2 at: http://mlflow:5000/#/experiments/1/runs/7f366e2bc1ad4abca419d49648b04df1
🧪 View experiment at: http://mlflow:5000/#/experiments/1


In [4]:
result = mlflow.register_model(
    model_uri="runs:/7f366e2bc1ad4abca419d49648b04df1/model",
    name="bonsai_diagnosis_model"
)
print(result)

Successfully registered model 'bonsai_diagnosis_model'.
2026/09/30 03:27:04 WARNING mlflow.tracking._model_registry.fluent: Run with id 7f366e2bc1ad4abca419d49648b04df1 has no artifacts at artifact path 'model', registering model based on models:/m-31b6590209d34683b32bce04897bc617 instead
2026/09/30 03:27:04 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: bonsai_diagnosis_model, version 1


<ModelVersion: aliases=[], creation_timestamp=1790738824372, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1790738824372, metrics=None, model_id=None, name='bonsai_diagnosis_model', params=None, run_id='7f366e2bc1ad4abca419d49648b04df1', run_link='', source='models:/m-31b6590209d34683b32bce04897bc617', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>


Created version '1' of model 'bonsai_diagnosis_model'.
